# Predict horizon classes for a list of prompts

Takes a list of plain-text prompts, caches the residual-stream activation at the
output of transformer layer 21 for each prompt's final token, and pushes those
vectors through the saved projection pipeline:

    activation -> PLS1-3 + residual PC1-3 -> per-folder shift -> (t, u) -> horizon class

Nothing is fitted here. Extraction lives in `vendor/activations/`; the projection
models are the artifacts under `models/`.

Two things differ from a plain surface projection. Both were chosen on scores
measured 5-fold grouped by task, so no task was shared between fitting and
scoring:

* **A per-folder translation in the PLS1-PLS2 plane** (the plane orthogonal to
  `u`) is applied before projecting. Prompt framing rigidly displaces the whole
  manifold, and removing that offset takes the overall surface RMSE from 23.8 to
  7.9. The shifts were fitted from coordinates alone, with no horizon labels, so
  a raw prompt with no `source_folder` is handled by recovering the folder
  geometrically -- which identifies `abst` with 98.8% accuracy and costs about
  0.0004 accuracy.
* **The feature set depends on the folder.** The reconstruction-residual PCs
  carry real signal for the concrete-task folders and are noise for `abst`, so
  `abst` is predicted from `t` alone.

One caveat on every number below: each point the classifiers were trained on is
an average over dozens of prompt variants sharing a task and a horizon, while a
prompt here is a single sample. Single prompts therefore sit further from the
fitted surface than a training point does, and are predicted less accurately.

## Setup

In [ ]:
# Reload edited modules automatically: the helpers under `vendor/` change often,
# and without this an already-imported module keeps its stale copy for the life
# of the kernel (an ImportError for a function that plainly exists on disk).
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# The saved models unpickle classes from `temporal_manifolds`; a minimal copy of
# that code lives in `vendor/` at the repo root.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vendor").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "vendor") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "vendor"))

from activations.extraction import (
    PROMPT_TOKEN_POSITION,
    TARGET_LAYER_COMPONENT,
    extract_last_position_activation,
    load_model_tokenizer,
)
from activations.residual_pca import load_residual_pca, project_activations
from temporal_manifolds.viz.activation_pls import load_pls_model
from temporal_manifolds.viz.extruded_surface import load_surface_model
from utils.horizon_classes import HORIZON_CLASS_LABELS, horizon_class
from utils.ordinal_regression import load_ordinal_model

print(f"repo root: {REPO_ROOT}")
print(f"target: {TARGET_LAYER_COMPONENT} @ position {PROMPT_TOKEN_POSITION}")

## Configuration

`MODEL_NAME` must be the model the PLS artifact was fit on -- the projection is
meaningless in another model's activation space; the artifact's own metadata is
checked against the loaded model below. `HF_TOKEN` is read from the environment
if left as `None`.

The curve and surface are the original shared artifacts; per-folder curves were
tried and did not generalise better. `DEGREE = 2` -- degree 3 was no better
anywhere and overfits more.

In [ ]:
import os

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
HF_TOKEN = os.environ.get("HF_TOKEN")  # or paste the token string here

DEVICE = None       # None -> cuda, else mps, else cpu
DTYPE = "bfloat16"  # None keeps the checkpoint dtype
SYSTEM_PROMPT = ""  # empty means no system turn is added, as in the cached runs
THINKING = False    # chat-template `enable_thinking`

OFFSET_DEGREE = 2   # extrusion profile degree, picks the surface artifact
DEGREE = 2          # classifier polynomial degree, picks the model artifacts

PLS_PATH = REPO_ROOT / "models" / "ctype_only_activation_pls_layer_out-21.joblib"
RESIDUAL_PCA_PATH = REPO_ROOT / "models" / "ctype_only_residual_pca_layer_out-21.joblib"
SURFACE_PATH = (
    REPO_ROOT / "models"
    / f"ctype_only_activation_surface_PLS1-PLS2-by-t_extruded-PLS3_degree-{OFFSET_DEGREE}.joblib"
)
SHIFT_TABLE_PATH = REPO_ROOT / "models" / "new_per_folder_shift_table.joblib"

# `abst` drops the residual PCs; the concrete-task folders keep them. Each
# artifact carries its own feature order, so it is not repeated here.
ABST_CLASSIFIER_PATH = (
    REPO_ROOT / "models"
    / f"new_horizon_class_abst_ordinal_binary-decomposition_degree-{DEGREE}.joblib"
)
MAIN_CLASSIFIER_PATH = (
    REPO_ROOT / "models"
    / f"new_horizon_class_main_ordinal_binary-decomposition_degree-{DEGREE}.joblib"
)

ABST_FOLDER = "abst"       # folder routed to the residual-free classifier
COORDINATE_COLUMNS = ["PLS1", "PLS2", "PLS3"]
PARAMETER_SAMPLES = 2000   # density of the nearest-t search when projecting

print(f"model: {MODEL_NAME}")
print(f"HF token: {'set' if HF_TOKEN else 'not set'}")

## Load the projection models

In [ ]:
import joblib

pls, pls_metadata = load_pls_model(PLS_PATH)
residual_pca, residual_metadata = load_residual_pca(RESIDUAL_PCA_PATH)
surface, surface_metadata = load_surface_model(SURFACE_PATH)

shift_table = joblib.load(SHIFT_TABLE_PATH)
SHIFTS = {name: np.asarray(value, dtype=float) for name, value in shift_table["shifts"].items()}
SHIFT_NAMES = sorted(SHIFTS)

abst_model, abst_features, abst_metadata = load_ordinal_model(ABST_CLASSIFIER_PATH)
main_model, main_features, main_metadata = load_ordinal_model(MAIN_CLASSIFIER_PATH)

print("pls       :", PLS_PATH.name)
print("           ", pls_metadata["layer_component"],
      "| position", pls_metadata["cached_position"],
      "| components", pls_metadata["component_count"],
      "| activation width", pls_metadata["feature_count"])
print("residual  :", RESIDUAL_PCA_PATH.name)
print("            components", residual_pca.components_.shape[0])
print("surface   :", SURFACE_PATH.name)
print("            offset degree", surface.offset_degree,
      "| t range", np.round(surface.training_parameter_bounds, 3))
print("shifts    :", SHIFT_TABLE_PATH.name)
for name in SHIFT_NAMES:
    print("            %-11s (%8.3f, %8.3f)" % (name, SHIFTS[name][0], SHIFTS[name][1]))
print("abst clf  :", ABST_CLASSIFIER_PATH.name, "->", abst_features)
print("main clf  :", MAIN_CLASSIFIER_PATH.name, "->", main_features)

## Prompts

Edit this list. Anything iterable of `str` works -- read them from a file if the
list gets long.

In [ ]:
PROMPTS = [
    "Could you help me plan how to write a wedding speech? I have 2 weeks available.",
    "Answer yes or no: is the sky blue?",
    "Draft a five-year research agenda for a new materials science lab.",
    "What should our civilisation prioritise over the next ten thousand years?",
    "Reply to this email in one sentence.",
]

for index, prompt in enumerate(PROMPTS):
    print(f"{index:>3}  {prompt}")
print(f"\n{len(PROMPTS)} prompt(s)")

## Load the language model

Downloads the checkpoint from the Hugging Face Hub on first run and reuses the
local cache afterwards. This is the only expensive cell in the notebook.

In [ ]:
model, tokenizer = load_model_tokenizer(
    MODEL_NAME,
    hf_token=HF_TOKEN,
    device=DEVICE,
    dtype=DTYPE,
    system_prompt=SYSTEM_PROMPT,
)

hidden_size = int(model.config.hidden_size)
if hidden_size != pls_metadata["feature_count"]:
    raise ValueError(
        f"{MODEL_NAME} has hidden size {hidden_size}, but the PLS artifact was fit on "
        f"{pls_metadata['feature_count']}-dimensional activations."
    )

print(f"layers: {model.config.num_hidden_layers}, hidden size: {hidden_size}")
print(f"device: {next(model.parameters()).device}, dtype: {next(model.parameters()).dtype}")

## Extract and project

One forward pass per prompt, then the coordinates the classifiers were trained
on: PLS scores and the principal components of what those scores fail to
reconstruct. The result is the same `df` the rest of the notebook expects, so
everything downstream is unchanged.

In [ ]:
records = []
for index, prompt in enumerate(PROMPTS):
    activation = extract_last_position_activation(model, tokenizer, prompt, thinking=THINKING)
    scores, residual_scores, residual_rms = project_activations(
        pls, residual_pca, activation.numpy()
    )
    records.append({
        "prompt_index": index,
        "prompt": prompt,
        "PLS1": float(scores[0, 0]),
        "PLS2": float(scores[0, 1]),
        "PLS3": float(scores[0, 2]),
        "reconstruction_residual_PC1": float(residual_scores[0, 0]),
        "reconstruction_residual_PC2": float(residual_scores[0, 1]),
        "reconstruction_residual_PC3": float(residual_scores[0, 2]),
        "reconstruction_residual_rms": float(residual_rms[0]),
    })
    print(f"{index:>3}  extracted  |activation| ok  PLS "
          f"({scores[0, 0]:8.2f}, {scores[0, 1]:8.2f}, {scores[0, 2]:8.2f})")

df = pd.DataFrame.from_records(records)
print(f"\n{len(df)} prompt(s) projected")
df.head()

## Assign the per-folder shift

`source_folder` is used when it names a folder in the table. Otherwise the shift
is inferred geometrically: each row takes the shift that puts it closest to the
surface. The shifts were fitted without horizon labels, so this inference uses no
information the model would not have at prediction time.

In [ ]:
def assign_shifts(frame):
    """Return (folder_label, shift) per row, inferring the folder when unlabelled."""

    coordinates = frame[COORDINATE_COLUMNS].to_numpy(float)
    labels = np.full(len(frame), None, dtype=object)

    if "source_folder" in frame.columns:
        known = frame["source_folder"].isin(SHIFT_NAMES).to_numpy()
        labels[known] = frame.loc[known, "source_folder"].to_numpy()

    unknown = np.array([label is None for label in labels])
    if unknown.any():
        # Distance to the surface under every candidate shift; take the nearest.
        candidates = np.stack([
            surface.project(
                np.column_stack([coordinates[unknown, :2] - SHIFTS[name],
                                 coordinates[unknown, 2]]),
                parameter_samples=PARAMETER_SAMPLES,
            )[2]
            for name in SHIFT_NAMES
        ])
        labels[unknown] = np.array(SHIFT_NAMES, dtype=object)[candidates.argmin(axis=0)]

    return labels, np.stack([SHIFTS[label] for label in labels])


folder_label, shift = assign_shifts(df)
df["folder_assigned"] = folder_label
if "source_folder" in df.columns:
    df["folder_inferred"] = ~df["source_folder"].isin(SHIFT_NAMES).to_numpy()
else:
    df["folder_inferred"] = True

print("folder taken from the file :", int((~df["folder_inferred"]).sum()), "row(s)")
print("folder inferred from geometry:", int(df["folder_inferred"].sum()), "row(s)")
print(df["folder_assigned"].value_counts().to_string())

## Predict

In [ ]:
# Surface coordinates: u is exactly PLS3, t is the nearest point along the
# extruded curve *after* the folder's translation has been removed.
shifted = df[COORDINATE_COLUMNS].to_numpy(float).copy()
shifted[:, :2] -= shift

t, u, distance = surface.project(shifted, parameter_samples=PARAMETER_SAMPLES)
df["t"] = t
df["u"] = u
df["surface_distance"] = distance

is_abst = (df["folder_assigned"] == ABST_FOLDER).to_numpy()

# Only the non-abst rows need the residual PCs, so demand them only if present.
needed = [column for column in main_features if column not in {"t", "u"}]
missing = [column for column in needed if column not in df.columns]
if (~is_abst).any() and missing:
    raise KeyError(
        f"the extracted features are missing column(s) needed for non-abst rows: {missing}"
    )

predicted = np.empty(len(df), dtype=int)
for mask, model, features in ((is_abst, abst_model, abst_features),
                              (~is_abst, main_model, main_features)):
    if mask.any():
        predicted[mask] = model.predict(df.loc[mask, features].to_numpy(float))

df["horizon_class_predicted"] = predicted
df["horizon_class_label_predicted"] = np.array(HORIZON_CLASS_LABELS, dtype=object)[predicted]

print("distance to surface: mean %.3f, max %.3f" % (distance.mean(), distance.max()))
outside = (t < surface.training_parameter_bounds[0]) | (t > surface.training_parameter_bounds[1])
print("points projecting outside the fitted t range:", int(outside.sum()))
print("routed to the abst classifier:", int(is_abst.sum()),
      "| to the main classifier:", int((~is_abst).sum()))

## Classifier scores

The classifier is a Frank-Hall binary decomposition, so it has **no per-class
logits**: it is `K - 1` independent logistic models, each answering "is the class
greater than k?". The raw scores are therefore *cumulative* log-odds, one column
per threshold -- `logit_gt_k` is the log-odds that the horizon exceeds the upper
edge of class `k`.

Read them as a monotone staircase: for a well-behaved row the scores start
positive and cross zero exactly once, at the predicted class. Because the eight
models are fitted independently, nothing forces the columns to decrease, and a
non-monotone row is a genuine disagreement between thresholds. `predict_proba`
resolves it by projecting onto a monotone sequence before differencing, so
comparing the raw scores against the probabilities shows where the model is
internally inconsistent.

Class probabilities are also written out. Those come from differencing the
monotone cumulative probabilities and are clipped at zero, so they can sum to
slightly under one; the normalised version is what `predict` uses.

In [ ]:
THRESHOLD_COUNT = len(HORIZON_CLASS_LABELS) - 1
LOGIT_COLUMNS = [f"logit_gt_{k}" for k in range(THRESHOLD_COUNT)]
PROBABILITY_COLUMNS = [f"p_{label}" for label in HORIZON_CLASS_LABELS]

logits = np.full((len(df), THRESHOLD_COUNT), np.nan)
probabilities = np.full((len(df), len(HORIZON_CLASS_LABELS)), np.nan)

for mask, model, features in ((is_abst, abst_model, abst_features),
                             (~is_abst, main_model, main_features)):
    if not mask.any():
        continue
    values = df.loc[mask, features].to_numpy(float)
    # Everything before the final step is the polynomial expansion and scaler;
    # the ordinal estimator itself holds the K-1 binary models.
    inner = model[:-1].transform(values)
    logits[mask] = model.named_steps["ordinal"].decision_function(inner)
    probabilities[mask] = model.predict_proba(values)

df[LOGIT_COLUMNS] = logits
df[PROBABILITY_COLUMNS] = probabilities

total = probabilities.sum(axis=1, keepdims=True)
normalized = np.divide(probabilities, np.where(total > 0, total, 1.0))
df["predicted_probability"] = normalized.max(axis=1)

# A row is monotone when the cumulative log-odds never rise with k, which is what
# the ordinal assumption expects.
non_monotone = (np.diff(logits, axis=1) > 0).any(axis=1)
df["logits_monotone"] = ~non_monotone

print("threshold k -> logit_gt_k is the log-odds that the class exceeds:")
for k in range(THRESHOLD_COUNT):
    print(f"   logit_gt_{k}  P(class > {k})  i.e. beyond '{HORIZON_CLASS_LABELS[k]}'")

print("\nrows with non-monotone cumulative logits: %d of %d (%.2f%%)"
      % (int(non_monotone.sum()), len(df), 100 * non_monotone.mean()))
print("mean predicted-class probability: %.4f" % df["predicted_probability"].mean())

print("\nlogit summary:")
print(df[LOGIT_COLUMNS].describe().loc[["mean", "std", "min", "max"]].round(3).to_string())

df[["folder_assigned", "horizon_class_label_predicted", "predicted_probability"]
   + LOGIT_COLUMNS].head()

## The surface, and the prompts projected onto it

The sheet is the fitted extruded surface, sampled across its `t` range and the
`u` span the prompts occupy. The markers are **not** the prompts' PLS
coordinates: each one is `S(t, u)`, the nearest point *on* the surface, so every
marker lies exactly on the sheet by construction. The distance that was thrown
away in getting there is `surface_distance`, shown on hover -- a large value
means the marker is a poor stand-in for where the prompt actually sits.

Everything is drawn in the shifted frame, with each prompt's folder translation
already removed, which is the frame the surface itself lives in.

In [ ]:
projected = surface.predict(df["t"].to_numpy(float), df["u"].to_numpy(float))
df[["PLS1_on_surface", "PLS2_on_surface", "PLS3_on_surface"]] = projected

# Sample the sheet over its fitted t range, and over a u span that covers the
# prompts with a little air around them.
u_values = df["u"].to_numpy(float)
u_lo, u_hi = float(np.min(u_values)), float(np.max(u_values))
if not np.isfinite([u_lo, u_hi]).all() or (u_hi - u_lo) < 1e-6:
    u_lo, u_hi = (float(v) for v in surface.training_extrusion_bounds)
pad = 0.15 * max(u_hi - u_lo, 1.0)
u_grid = np.linspace(u_lo - pad, u_hi + pad, 60)
t_grid = np.linspace(*surface.training_parameter_bounds, 200)

grid_x, grid_y, grid_z = surface.grid(t_grid, u_grid)

figure = go.Figure()
figure.add_surface(
    x=grid_x, y=grid_y, z=grid_z,
    surfacecolor=np.broadcast_to(t_grid[:, None], grid_x.shape),
    colorscale="Viridis", opacity=0.55, showscale=True,
    colorbar={"title": "t"},
    name="fitted surface",
    hovertemplate="t=%{surfacecolor:.3f}<br>PLS1=%{x:.2f}<br>PLS2=%{y:.2f}<br>PLS3=%{z:.2f}<extra></extra>",
)
figure.add_scatter3d(
    x=df["PLS1_on_surface"], y=df["PLS2_on_surface"], z=df["PLS3_on_surface"],
    mode="markers+text",
    marker={"size": 6, "color": "crimson", "line": {"width": 1, "color": "white"}},
    text=df["prompt_index"].astype(str),
    textposition="top center",
    name="prompts projected onto the surface",
    customdata=np.column_stack([
        df["prompt"].str.slice(0, 60),
        df["t"], df["u"], df["surface_distance"],
        df["horizon_class_label_predicted"],
        df["folder_assigned"],
    ]),
    hovertemplate=(
        "%{customdata[0]}<br>t=%{customdata[1]:.3f}  u=%{customdata[2]:.2f}"
        "<br>distance to surface=%{customdata[3]:.2f}"
        "<br>predicted: %{customdata[4]}"
        "<br>shift used: %{customdata[5]}<extra></extra>"
    ),
)
figure.update_layout(
    scene={"xaxis_title": "PLS1 (shifted)",
           "yaxis_title": "PLS2 (shifted)",
           "zaxis_title": "PLS3 = u"},
    title="Prompts projected onto the fitted surface",
    autosize=True, width=None, height=700,
    legend={"orientation": "h", "yanchor": "bottom", "y": -0.08},
)
figure.show()

print("distance discarded by the projection: mean %.3f, max %.3f"
      % (df["surface_distance"].mean(), df["surface_distance"].max()))

## Results

In [ ]:
columns = [
    "folder_assigned", "t", "u", "surface_distance",
    "horizon_class_predicted", "horizon_class_label_predicted",
]

# Score the predictions when the file carries the ground truth.
if "time_horizon_months" in df.columns:
    df["horizon_class"] = horizon_class(df["time_horizon_months"])
    df["horizon_class_label"] = np.array(
        list(HORIZON_CLASS_LABELS) + ["< 1 second"], dtype=object
    )[np.where(df["horizon_class"] < 0, len(HORIZON_CLASS_LABELS), df["horizon_class"])]
    known = df["horizon_class"] >= 0
    error = (
        df.loc[known, "horizon_class_predicted"] - df.loc[known, "horizon_class"]
    ).abs()
    print("scored on %d row(s) with a known horizon:" % int(known.sum()))
    print("  accuracy            %.4f" % (error == 0).mean())
    print("  within-one accuracy %.4f" % (error <= 1).mean())
    print("  mean absolute error %.4f classes" % error.mean())

    scored = df[known]
    by_folder = scored.groupby("folder_assigned")
    per_folder = pd.DataFrame({
        "n": by_folder.size(),
        "accuracy": (scored["horizon_class_predicted"] == scored["horizon_class"])
            .groupby(scored["folder_assigned"]).mean(),
        "within_one": ((scored["horizon_class_predicted"] - scored["horizon_class"]).abs() <= 1)
            .groupby(scored["folder_assigned"]).mean(),
        "surface_rmse": by_folder["surface_distance"].apply(
            lambda s: np.sqrt((s ** 2).mean())),
    })
    print("\nper folder:")
    print(per_folder.round(4).to_string())

    columns = ["horizon_class", "horizon_class_label"] + columns

summary = df["horizon_class_label_predicted"].value_counts().reindex(
    HORIZON_CLASS_LABELS, fill_value=0
)
print("\npredicted class counts:")
print(summary.to_string())

df[columns]

## Secondary scoring: the 7-class scheme

The same predictions, scored after merging `day - week`, `week - month` and
`month - year` into a single `day - year` bucket. Nothing is refitted and no
prediction changes -- the 9-class output is simply read at a coarser resolution.

This particular merge is not chosen to flatter the number. Those three classes
are the ones measured to be inseparable in `t`: their adjacent-class overlap runs
0.47 to 0.67 for `abst`, and the median gap between neighbouring class centres is
about 0.007 against a within-class spread of 0.064. Collapsing them declines a
distinction the activations do not support.

Note that merging *any* classes raises raw accuracy, because the majority-class
baseline rises too, so the baseline and Cohen's kappa are printed alongside.
Merging further -- to 5, 3 or 2 classes -- keeps inflating accuracy while
quadratic-weighted kappa falls, which is why only this one merge is reported.

In [ ]:
from sklearn.metrics import cohen_kappa_score

from utils.horizon_classes import (
    MERGED_HORIZON_CLASS_LABELS,
    merge_horizon_classes,
)

if "time_horizon_months" in df.columns:
    merged_true = merge_horizon_classes(df["horizon_class"].to_numpy())
    merged_pred = merge_horizon_classes(df["horizon_class_predicted"].to_numpy())
    df["horizon_class_merged"] = merged_true
    df["horizon_class_merged_predicted"] = merged_pred
    df["horizon_class_label_merged_predicted"] = np.array(
        MERGED_HORIZON_CLASS_LABELS, dtype=object
    )[merged_pred]

    scored = df["horizon_class"] >= 0
    true_7 = merged_true[scored.to_numpy()]
    pred_7 = merged_pred[scored.to_numpy()]
    error_7 = np.abs(pred_7 - true_7)
    baseline_7 = pd.Series(true_7).value_counts(normalize=True).max()

    print("7-class scheme, scored on %d row(s):" % len(true_7))
    print("  accuracy            %.4f" % (error_7 == 0).mean())
    print("  majority baseline   %.4f" % baseline_7)
    print("  within-one accuracy %.4f" % (error_7 <= 1).mean())
    print("  mean absolute error %.4f classes" % error_7.mean())
    print("  kappa               %.4f" % cohen_kappa_score(true_7, pred_7))
    print("  kappa (quadratic)   %.4f"
          % cohen_kappa_score(true_7, pred_7, weights="quadratic"))

    folders = df.loc[scored, "folder_assigned"].to_numpy()
    per_folder_7 = pd.DataFrame({
        "n": pd.Series(true_7).groupby(folders).size(),
        "accuracy": pd.Series(pred_7 == true_7).groupby(folders).mean(),
        "within_one": pd.Series(error_7 <= 1).groupby(folders).mean(),
    })
    print("\nper folder (7-class):")
    print(per_folder_7.round(4).to_string())

    print("\npredicted class counts (7-class):")
    print(
        df["horizon_class_label_merged_predicted"]
        .value_counts()
        .reindex(MERGED_HORIZON_CLASS_LABELS, fill_value=0)
        .to_string()
    )
else:
    print("no time_horizon_months column: nothing to score")